## Creating and Aligning Keypoints with Respiration + Behavior

In [1]:
import os
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import keypoint_moseq as kpms

### RI1 + RI2 sleap, respiration, and behavior folderpaths

In [2]:
sleap_h5s_path = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\Tanish\Keypoint RI1+RI2"
resp_h5s_path = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\Tanish\Keypoint RI1+RI2\Resp_h5"
behavior_boris_path = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\boris_csv_RI1 and RI2"

### 1) Discover SLEAP/respiration files + inspect metadata

In [3]:
def find_h5_files(base_path):
    """Recursively find H5/HDF5 files under a directory."""
    base = Path(base_path)
    if not base.exists():
        return []

    patterns = ("*.h5", "*.hdf5", "*.H5", "*.HDF5")
    files = []
    for pattern in patterns:
        files.extend(base.rglob(pattern))
    return sorted(set(files))


def inspect_h5(file_path, max_items=20):
    """Return dataset metadata from an h5 file."""
    summary = []
    with h5py.File(file_path, "r") as h5:
        def visitor(name, obj):
            if len(summary) >= max_items:
                return
            if isinstance(obj, h5py.Dataset):
                summary.append({"name": name, "shape": obj.shape, "dtype": str(obj.dtype)})
        h5.visititems(visitor)
    return summary


def infer_h5_kind(file_path):
    """Classify h5 as likely SLEAP, respiration, or keypoint-moseq artifact."""
    with h5py.File(file_path, "r") as h5:
        keys = set(h5.keys())

        if "model_snapshots" in keys or "data" in keys:
            return "kpms_artifact"
        if {"resp", "ekg"}.issubset(keys):
            return "respiration"

        required = {"tracks", "point_scores", "node_names"}
        if required.issubset(keys):
            return "sleap"

    return "unknown"


def is_valid_sleap_for_kpms(file_path):
    """True only for SLEAP analysis files accepted by kpms.load_keypoints(..., 'sleap')."""
    try:
        return infer_h5_kind(file_path) == "sleap"
    except Exception:
        return False


all_sleap_path_h5s = find_h5_files(sleap_h5s_path)
resp_files = find_h5_files(resp_h5s_path)

# Keep only valid SLEAP analysis files and explicitly ignore respiration subfolder.
sleap_files = [
    f for f in all_sleap_path_h5s
    if "resp_h5" not in str(f).lower() and is_valid_sleap_for_kpms(f)
]

print(f"Found {len(sleap_files)} SLEAP candidate file(s)")
print(f"Found {len(resp_files)} respiration file(s)")

if sleap_files:
    print("\nSLEAP examples:")
    for f in sleap_files[:5]:
        print(f"- {f.name}")
else:
    print("\nNo SLEAP files found. Confirm sleap_h5s_path points to your exported SLEAP .h5 files.")

if resp_files:
    print("\nRespiration examples:")
    for f in resp_files[:5]:
        print(f"- {f.name}")
else:
    print("\nNo respiration files found. Confirm resp_h5s_path points to the folder with respiration .h5 files.")

sleap_preview = inspect_h5(sleap_files[0]) if sleap_files else []
resp_preview = inspect_h5(resp_files[0]) if resp_files else []

print("\nSLEAP preview (first file):")
for item in sleap_preview[:10]:
    print(item)

print("\nRespiration preview (first file):")
for item in resp_preview[:10]:
    print(item)

Found 17 SLEAP candidate file(s)
Found 18 respiration file(s)

SLEAP examples:
- top-RI1_s1_1_p5_2_nRB6_20250622_143958.1.analysis.h5
- top-RI1_s1_2_p5_1_nRB6_20250622_170742.1.analysis.h5
- top-RI1_s2_3_p5_3_nRB3_20250622_104059.1.analysis.h5
- top-RI1_s2_4_p5_4_nRB3_20250622_123424.1.analysis.h5
- top-RI1_s3_6_p5_3_nRB3_20250621_125312.1.analysis.h5

Respiration examples:
- RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5
- RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5
- RI1_s2_3_p5_3_nRB3_20250622_104059_merged.h5
- RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5
- RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5

SLEAP preview (first file):
{'name': 'edge_inds', 'shape': (13, 2), 'dtype': 'int32'}
{'name': 'edge_names', 'shape': (13, 2), 'dtype': '|S12'}
{'name': 'instance_scores', 'shape': (2, 9300), 'dtype': 'float64'}
{'name': 'labels_path', 'shape': (), 'dtype': 'object'}
{'name': 'node_names', 'shape': (8,), 'dtype': '|S12'}
{'name': 'point_scores', 'shape': (2, 8, 9300), 'dtype': 'float6

### 2) BORIS behavior helpers (adapted for clean extraction)

In [4]:
def threshold_bouts(start_stop_array, min_iti=0, min_bout=0):
    """Merge close bouts and remove short bouts (times in seconds)."""
    if start_stop_array.size == 0:
        return np.empty((0, 2))

    bouts = np.array(start_stop_array, dtype=float)
    bouts = bouts[np.argsort(bouts[:, 0])]

    merged = [bouts[0].copy()]
    for start, stop in bouts[1:]:
        if start - merged[-1][1] < min_iti:
            merged[-1][1] = max(merged[-1][1], stop)
        else:
            merged.append(np.array([start, stop], dtype=float))

    merged = np.array(merged)
    durations = merged[:, 1] - merged[:, 0]
    return merged[durations >= min_bout]


def get_behavior_bouts(boris_df, subject, behavior, min_iti=0, min_bout=0):
    """Extract behavior bout start/stop times from BORIS and return milliseconds."""
    start_stop_arrays = []

    for mouse in subject:
        subject_df = boris_df[boris_df["Subject"] == mouse]
        behavior_arrays = []
        for act in behavior:
            behavior_df = subject_df[subject_df["Behavior"] == act]
            start_stop_array = behavior_df[["Start (s)", "Stop (s)"]].to_numpy()
            if start_stop_array.size > 0:
                behavior_arrays.append(start_stop_array)

        if behavior_arrays:
            start_stop_array = np.concatenate(behavior_arrays)
            start_stop_arrays.append(threshold_bouts(start_stop_array, min_iti, min_bout))

    if not start_stop_arrays:
        return np.empty((0, 2))

    start_stop_array = np.concatenate(start_stop_arrays)
    organizer = np.argsort(start_stop_array[:, 0])
    start_stop_array = start_stop_array[organizer]
    return start_stop_array * 1000

### 3) Load BORIS CSVs and extract behavior bouts

In [5]:
def normalize_recording_id(stem):
    """Normalize IDs so SLEAP/BORIS/resp filenames match."""
    clean = stem.replace("_merged", "")
    clean = re.sub(r"^top-", "", clean)
    clean = re.sub(r"\.analysis$", "", clean)

    # BORIS exports may end with .1, .2, etc.
    if "." in clean and clean.rsplit(".", 1)[-1].isdigit():
        clean = clean.rsplit(".", 1)[0]

    for suffix in ("_ss",):
        if clean.endswith(suffix):
            clean = clean[: -len(suffix)]
    return clean


def load_boris_tables(boris_dir):
    """Load all BORIS CSVs in a directory into a dict[file_stem] = DataFrame."""
    boris_path = Path(boris_dir)
    csv_files = sorted(boris_path.glob("*.csv"))
    boris_tables = {}

    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        for col in ["Image index start", "Image index stop"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        boris_tables[csv_file.stem] = df

    return boris_tables


boris_tables = load_boris_tables(behavior_boris_path)
print(f"Loaded {len(boris_tables)} BORIS file(s)")

if not boris_tables:
    print("No BORIS CSV files found. Confirm behavior_boris_path.")
else:
    first_key = next(iter(boris_tables))
    first_df = boris_tables[first_key]
    print(f"\nExample file: {first_key}")
    print(f"Rows: {len(first_df)}")
    print("Subjects:", sorted(first_df["Subject"].dropna().astype(str).unique()))
    print("Behaviors:", sorted(first_df["Behavior"].dropna().astype(str).unique()))


def recording_core_id(rec_id):
    """Session-level ID without timestamp for fallback matching."""
    # Handle both p5_3 and p_5_3 style naming.
    match = re.search(r"(RI\d+_s\d+_\d+_p_?\d+_\d+_nRB\d+)", rec_id)
    return match.group(1) if match else rec_id


# BORIS is the FPS source for this dataset.
boris_fps_map = {}
boris_fps_core_map = {}
boris_fps_debug = {}

for stem, df in boris_tables.items():
    fps_col = next((c for c in df.columns if "FPS" in c.upper()), None)
    if fps_col is None:
        continue

    fps_vals = pd.to_numeric(df[fps_col], errors="coerce").dropna().unique()
    if len(fps_vals) == 0:
        continue

    rec_id = normalize_recording_id(stem)
    chosen_fps = float(np.median(fps_vals))

    boris_fps_map[rec_id] = chosen_fps
    boris_fps_debug[rec_id] = np.asarray(fps_vals, dtype=float)

    core_id = recording_core_id(rec_id)
    if core_id not in boris_fps_core_map:
        boris_fps_core_map[core_id] = chosen_fps

print(f"\nExtracted BORIS FPS for {len(boris_fps_map)} recording(s)")


# Match each SLEAP recording to BORIS FPS after normalization.
# Pass 1: exact recording id; Pass 2: core-id fallback.
fps_map_merged = {}
fps_match_source = {}
for f in sleap_files:
    rec_id = normalize_recording_id(Path(f).stem)

    fps = boris_fps_map.get(rec_id)
    source = "exact"
    if fps is None:
        fps = boris_fps_core_map.get(recording_core_id(rec_id))
        source = "core"

    fps_map_merged[rec_id] = fps
    fps_match_source[rec_id] = source if fps is not None else "unmatched"

resolved = sum(v is not None for v in fps_map_merged.values())
print(f"Resolved FPS for {resolved}/{len(fps_map_merged)} SLEAP recordings")

missing_fps_ids = sorted([rid for rid, fps in fps_map_merged.items() if fps is None])
if missing_fps_ids:
    print("\nRecordings without matched FPS (showing up to 12):")
    for rid in missing_fps_ids[:12]:
        print(f"- {rid}")

known_fps_values = [fps for fps in fps_map_merged.values() if fps is not None]
default_fps_fallback = float(np.median(known_fps_values)) if known_fps_values else 30.0
print(f"\nDefault FPS fallback for unmatched recordings: {default_fps_fallback:.6f}")


# Choose target labels to extract (edit these as needed)
target_subjects = ["subject"]
target_behaviors = ["facial sniffing", "body sniffing", "anogenital sniffing"]

boris_bouts_ms = {}
for file_id, df in boris_tables.items():
    available_subjects = df["Subject"].dropna().astype(str).unique().tolist()
    available_behaviors = df["Behavior"].dropna().astype(str).unique().tolist()

    matched_subjects = [s for s in available_subjects if s.lower() in {x.lower() for x in target_subjects}]
    matched_behaviors = [b for b in available_behaviors if b.lower() in {x.lower() for x in target_behaviors}]

    if not matched_subjects or not matched_behaviors:
        continue

    bouts_ms = get_behavior_bouts(
        df,
        subject=matched_subjects,
        behavior=matched_behaviors,
        min_iti=0,
        min_bout=0,
    )
    boris_bouts_ms[file_id] = bouts_ms

print(f"\nExtracted bouts for {len(boris_bouts_ms)} file(s)")
if boris_bouts_ms:
    example_file = next(iter(boris_bouts_ms))
    print(f"Example extracted array ({example_file}):")
    print(boris_bouts_ms[example_file][:10])

Loaded 17 BORIS file(s)

Example file: RI1_s1_1_p5_2_nRB6_20250622_143958.1
Rows: 92
Subjects: ['social_agent', 'subject']
Behaviors: ['anogenital sniffing', 'body sniffing', 'facial sniffing']

Extracted BORIS FPS for 17 recording(s)
Resolved FPS for 17/17 SLEAP recordings

Default FPS fallback for unmatched recordings: 30.000000

Extracted bouts for 17 file(s)
Example extracted array (RI1_s1_1_p5_2_nRB6_20250622_143958.1):
[[ 6846. 11077.]
 [13231. 17154.]
 [17385. 28385.]
 [29538. 33769.]
 [36231. 37462.]
 [37615. 39538.]
 [40923. 42077.]
 [42462. 43923.]
 [44385. 47769.]
 [48615. 53923.]]


### 4) Diagnose FPS joins

### Using boris fps for sleap

### Some recs have 13 or 15 fps in the boris, verify this is correct

In [6]:
print("SLEAP normalized IDs:")
for f in sleap_files:
    print(f"  '{normalize_recording_id(Path(f).stem)}'")

print("\nBORIS normalized IDs:")
for stem in boris_tables.keys():
    print(f"  '{normalize_recording_id(stem)}'")

print("\nBORIS FPS values by file:")
for stem, df in boris_tables.items():
    fps_col = next((c for c in df.columns if "FPS" in c.upper()), None)
    if fps_col is None:
        print(f"{stem}: no FPS column")
        continue
    fps_vals = pd.to_numeric(df[fps_col], errors="coerce").dropna().unique()
    print(f"{stem}: {fps_vals}")

print("\nSLEAP -> matched FPS + source:")
for rec_id in sorted(fps_map_merged.keys()):
    print(f"{rec_id}: {fps_map_merged[rec_id]} ({fps_match_source[rec_id]})")

resolved = sum(v is not None for v in fps_map_merged.values())
print(f"\nResolved now: {resolved}/{len(fps_map_merged)}")
if resolved != len(fps_map_merged):
    unresolved = [rid for rid, fps in fps_map_merged.items() if fps is None]
    print("Unresolved IDs:")
    for rid in unresolved:
        print(f"- {rid}")
else:
    print("All SLEAP recordings resolved to BORIS FPS.")

SLEAP normalized IDs:
  'RI1_s1_1_p5_2_nRB6_20250622_143958'
  'RI1_s1_2_p5_1_nRB6_20250622_170742'
  'RI1_s2_3_p5_3_nRB3_20250622_104059'
  'RI1_s2_4_p5_4_nRB3_20250622_123424'
  'RI1_s3_6_p5_3_nRB3_20250621_125312'
  'RI1_s4_7_p5_2_nRB3_20250621_150707'
  'RI1_s4_8_p5_1_nRB3_20250621_165214'
  'RI1_s5_13_p5_4_nRB3_20250622_181801'
  'RI2_s1_1_p5_2_nRB6_20250622_150457'
  'RI2_s1_2_p5_1_nRB6_20250622_173049'
  'RI2_s2_3_p5_3_nRB3_20250622_110216'
  'RI2_s2_4_p5_4_nRB3_20250622_125648'
  'RI2_s3_5_p5_4_nRB3_20250621_112618'
  'RI2_s3_6_p_5_3_nRB3_20250621_131158'
  'RI2_s4_7_p5_2_nRB3_20250621_152519'
  'RI2_s4_8_p5_1_nRB3_20250621_171318'
  'RI2_s5_13_p5_4_nRB3_20250622_184136'

BORIS normalized IDs:
  'RI1_s1_1_p5_2_nRB6_20250622_143958'
  'RI1_s1_2_p5_1_nRB6_20250622_170742'
  'RI1_s2_3_p5_3_nRB3_20250622_104059'
  'RI1_s2_4_p5_4_nRB3_20250622_123424'
  'RI1_s3_6_p5_3_nRB3_HEEPS'
  'RI1_s4_7_p5_2_nRB3_HEEPS'
  'RI1_s4_8_p5_1_nRB3_HEEPS'
  'RI1_s5_13_p5_4_nRB3_20250622_181801'
  'RI2

### 5) Keypoint-MoSeq setup + training (dynamic FPS + dynamic bodyparts)

In [7]:
def decode_bodyparts(bodyparts):
    decoded = []
    for bp in bodyparts:
        if isinstance(bp, (bytes, np.bytes_)):
            decoded.append(bp.decode("utf-8", errors="ignore"))
        else:
            decoded.append(str(bp))
    return decoded


def choose_axis_bodyparts(bodyparts):
    """Pick anterior/posterior from available node names; fallback safely."""
    lowered = [bp.lower() for bp in bodyparts]

    anterior_candidates = ("nose", "snout", "head")
    posterior_candidates = ("tail_base", "tailbase", "tail", "hip", "rear")

    anterior = None
    posterior = None

    for token in anterior_candidates:
        if token in lowered:
            anterior = bodyparts[lowered.index(token)]
            break

    for token in posterior_candidates:
        if token in lowered:
            posterior = bodyparts[lowered.index(token)]
            break

    if anterior is None:
        anterior = bodyparts[0]
    if posterior is None:
        posterior = bodyparts[-1]

    return anterior, posterior


kpms_input_files = [str(p) for p in sleap_files]

if len(kpms_input_files) == 0:
    raise FileNotFoundError(
        "No usable SLEAP analysis .h5 files found. "
        "Expected keys: tracks, point_scores, node_names."
    )

# Use a project directory inside your current SLEAP folder
project_dir = str(Path(sleap_h5s_path) / "kpms_project")
os.makedirs(project_dir, exist_ok=True)

print(f"Using {len(kpms_input_files)} SLEAP file(s) for keypoint-moseq")
print(f"Project directory: {project_dir}")

# Load keypoints first so all config choices come from actual data attributes
coordinates, confidences, bodyparts_raw = kpms.load_keypoints(kpms_input_files, "sleap")
bodyparts = decode_bodyparts(bodyparts_raw)

anterior_bodypart, posterior_bodypart = choose_axis_bodyparts(bodyparts)
print("\nDetected bodyparts:", bodyparts)
print(f"Anterior bodypart: {anterior_bodypart}")
print(f"Posterior bodypart: {posterior_bodypart}")

# Build per-recording fps dict for auditability and reproducibility
recording_fps = {}
for f in kpms_input_files:
    rec_id = normalize_recording_id(Path(f).stem)
    recording_fps[rec_id] = fps_map_merged.get(rec_id)

resolved_fps = [v for v in recording_fps.values() if v is not None and v > 0]
fps_for_model = float(np.median(resolved_fps)) if resolved_fps else default_fps_fallback

print(f"\nResolved recording FPS for {len(resolved_fps)}/{len(recording_fps)} recordings")
print(f"Model FPS (median of resolved per-recording FPS): {fps_for_model:.6f}")

# Print a preview of the per-recording FPS map
for rec_id, fps in list(recording_fps.items())[:10]:
    print(f"- {rec_id}: {fps}")

# Create project structure once; if directory exists without config, force re-init.
if not Path(project_dir, "config.yml").exists():
    kpms.setup_project(project_dir, sleap_file=kpms_input_files[0], overwrite=True)

# Update config using discovered node names + measured FPS
kpms.update_config(
    project_dir,
    video_dir=str(sleap_h5s_path),
    anterior_bodyparts=[anterior_bodypart],
    posterior_bodyparts=[posterior_bodypart],
    use_bodyparts=bodyparts,
    fps=fps_for_model,
)

config = lambda: kpms.load_config(project_dir)

# Noise calibration
kpms.noise_calibration(project_dir, coordinates, confidences, **config())
plt.close("all")

# PCA + formatting
data, metadata = kpms.format_data(coordinates, confidences, **config())
pca = kpms.fit_pca(**data, **config())
kpms.save_pca(pca, project_dir)
kpms.print_dims_to_explain_variance(pca, 0.9)

# Optional diagnostics
kpms.plot_scree(pca, project_dir=project_dir)
kpms.plot_pcs(pca, project_dir=project_dir, **config())

# Fit AR-only stage
kpms.update_config(project_dir, latent_dim=4)
model = kpms.init_model(data, pca=pca, **config())
model = kpms.update_hypparams(model, kappa=5000)

model, model_name = kpms.fit_model(
    model,
    data,
    metadata,
    project_dir,
    ar_only=True,
    num_iters=50,
)

# Fit full ARHMM stage
model, data, metadata, current_iter = kpms.load_checkpoint(project_dir, model_name, iteration=50)
model = kpms.update_hypparams(model, kappa=1e4)
model = kpms.fit_model(
    model,
    data,
    metadata,
    project_dir,
    model_name,
    ar_only=False,
    start_iter=current_iter,
    num_iters=current_iter + 500,
)[0]

# Reindex + extract results
kpms.reindex_syllables_in_checkpoint(project_dir, model_name)
model, data, metadata, current_iter = kpms.load_checkpoint(project_dir, model_name)
results = kpms.extract_results(model, metadata, project_dir, model_name)

# Summarize top syllables
all_syllables = np.concatenate([v["syllable"] for v in results.values()])
valid_syllables = all_syllables[~np.isnan(all_syllables)]
valid_syllables = valid_syllables[valid_syllables >= 0].astype(int)

syllable_counts = Counter(valid_syllables)
top_syllables = [s for s, _ in syllable_counts.most_common(20)]

print("\nTop 20 most common syllables (RI1 + RI2 combined):")
for s in top_syllables:
    print(f"  Syllable {s}: {syllable_counts[s]} frames")

# Optional trajectory plots
kpms.generate_trajectory_plots(
    coordinates,
    results,
    project_dir,
    model_name,
    syllables=top_syllables,
    **config(),
)

Using 17 SLEAP file(s) for keypoint-moseq
Project directory: C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\Tanish\Keypoint RI1+RI2\kpms_project


Loading keypoints: 100%|████████████████| 17/17 [00:00<00:00, 44.72it/s]



Detected bodyparts: ['nose', 'head', 'left_ear', 'right_ear', 'middle', 'left_mid', 'right_mid', 'tail_base']
Anterior bodypart: nose
Posterior bodypart: tail_base

Resolved recording FPS for 17/17 recordings
Model FPS (median of resolved per-recording FPS): 30.000000
- RI1_s1_1_p5_2_nRB6_20250622_143958: 13.0
- RI1_s1_2_p5_1_nRB6_20250622_170742: 30.0
- RI1_s2_3_p5_3_nRB3_20250622_104059: 30.0
- RI1_s2_4_p5_4_nRB3_20250622_123424: 15.0
- RI1_s3_6_p5_3_nRB3_20250621_125312: 29.0
- RI1_s4_7_p5_2_nRB3_20250621_150707: 30.0
- RI1_s4_8_p5_1_nRB3_20250621_165214: 30.0
- RI1_s5_13_p5_4_nRB3_20250622_181801: 30.0
- RI2_s1_1_p5_2_nRB6_20250622_150457: 15.0
- RI2_s1_2_p5_1_nRB6_20250622_173049: 30.0


Loading sample frames: 100%|████████████| 30/30 [00:01<00:00, 21.98it/s]


KeyboardInterrupt: 

### 6) Verify respiration sampling rate


In [ ]:
def build_resp_file_map(resp_file_paths):
    file_map = {}
    for path in resp_file_paths:
        rec_id = normalize_recording_id(Path(path).stem)
        file_map[rec_id] = str(path)
    return file_map


def build_sleap_file_map(sleap_file_paths):
    file_map = {}
    for path in sleap_file_paths:
        rec_id = normalize_recording_id(Path(path).stem)
        file_map[rec_id] = str(path)
    return file_map


def estimate_resp_sampling_rate(resp_path, sleap_path, keypoint_fps):
    with h5py.File(resp_path, "r") as h5:
        if "resp" not in h5:
            return None
        resp_n_samples = int(np.asarray(h5["resp"]).shape[0])

    with h5py.File(sleap_path, "r") as h5:
        if "tracks" not in h5:
            return None
        n_frames = int(h5["tracks"].shape[-1])

    duration_s = n_frames / keypoint_fps if keypoint_fps and keypoint_fps > 0 else np.nan
    if not np.isfinite(duration_s) or duration_s <= 0:
        return None

    return resp_n_samples / duration_s


resp_file_map = build_resp_file_map(resp_files)
sleap_file_map = build_sleap_file_map(sleap_files)

sampling_rows = []
for rec_id, sleap_path in sleap_file_map.items():
    resp_path = resp_file_map.get(rec_id)
    rec_fps = fps_map_merged.get(rec_id)

    if resp_path is None or rec_fps is None:
        continue

    inferred_sr = estimate_resp_sampling_rate(resp_path, sleap_path, rec_fps)
    if inferred_sr is None:
        continue

    sampling_rows.append(
        {
            "recording_id": rec_id,
            "keypoint_fps": rec_fps,
            "inferred_resp_sampling_rate_hz": inferred_sr,
        }
    )

sampling_rate_check_df = pd.DataFrame(sampling_rows).sort_values("recording_id")

if sampling_rate_check_df.empty:
    raise RuntimeError("Could not infer respiration sampling rate from current files.")

resp_sampling_rate_hz_verified = float(sampling_rate_check_df["inferred_resp_sampling_rate_hz"].median())
sampling_rate_check_df["delta_from_median_hz"] = (
    sampling_rate_check_df["inferred_resp_sampling_rate_hz"] - resp_sampling_rate_hz_verified
)

print(f"Inferred respiration sampling rate (median): {resp_sampling_rate_hz_verified:.3f} Hz")
print("\nPer-recording inferred rates (first 10):")
print(sampling_rate_check_df.head(10).to_string(index=False))

### 7) Build final analysis dataframe (syllables + respiration + BORIS behavior)

In [ ]:
if "results" not in globals():
    raise RuntimeError("Run the keypoint-moseq training cell first so `results` is available.")


def behavior_col_name(behavior_name):
    safe = re.sub(r"[^a-zA-Z0-9]+", "_", str(behavior_name).strip().lower()).strip("_")
    return f"boris_{safe}"


def build_boris_table_map(boris_tables_dict):
    out = {}
    for stem, df in boris_tables_dict.items():
        rec_id = normalize_recording_id(stem)
        out[rec_id] = df
    return out


boris_table_map = build_boris_table_map(boris_tables)
analysis_frames = []

for result_key, result_payload in results.items():
    rec_id = normalize_recording_id(result_key)

    rec_fps = fps_map_merged.get(rec_id, default_fps_fallback)
    resp_path = resp_file_map.get(rec_id)
    boris_df = boris_table_map.get(rec_id)

    # Fallback to core-match BORIS table if exact id does not exist.
    if boris_df is None:
        rec_core = recording_core_id(rec_id)
        for candidate_id, candidate_df in boris_table_map.items():
            if recording_core_id(candidate_id) == rec_core:
                boris_df = candidate_df
                break

    if resp_path is None or boris_df is None:
        continue

    syllables = np.asarray(result_payload["syllable"]).squeeze()
    n_frames = len(syllables)
    if n_frames == 0:
        continue

    frame_idx = np.arange(n_frames)
    frame_time_s = frame_idx / rec_fps

    with h5py.File(resp_path, "r") as h5:
        if "resp" not in h5:
            continue
        resp_signal = np.asarray(h5["resp"][:]).squeeze()

    resp_idx = np.clip((frame_time_s * resp_sampling_rate_hz_verified).astype(int), 0, len(resp_signal) - 1)
    resp_at_frame = resp_signal[resp_idx]

    rec_df = pd.DataFrame(
        {
            "recording_id": rec_id,
            "frame_idx": frame_idx,
            "time_s": frame_time_s,
            "keypoint_fps": rec_fps,
            "resp": resp_at_frame,
            "syllable": syllables,
        }
    )

    subject_mask = boris_df["Subject"].astype(str).str.lower().isin({s.lower() for s in target_subjects})
    boris_subject_df = boris_df.loc[subject_mask].copy()

    unique_behaviors = sorted(boris_subject_df["Behavior"].dropna().astype(str).unique())
    behavior_cols = {}

    for behavior_name in unique_behaviors:
        col = behavior_col_name(behavior_name)
        is_behavior = np.zeros(n_frames, dtype=bool)

        bouts = boris_subject_df[boris_subject_df["Behavior"] == behavior_name]
        for _, row in bouts.iterrows():
            start_s = float(row["Start (s)"])
            stop_s = float(row["Stop (s)"])
            is_behavior |= (frame_time_s >= start_s) & (frame_time_s < stop_s)

        rec_df[col] = is_behavior
        behavior_cols[behavior_name] = col

    rec_df["boris_behavior_label"] = "none"
    for behavior_name, col in behavior_cols.items():
        rec_df.loc[rec_df[col], "boris_behavior_label"] = behavior_name

    analysis_frames.append(rec_df)

if not analysis_frames:
    raise RuntimeError("No aligned recordings were produced. Check `results`, BORIS, and resp file mappings.")

analysis_df = pd.concat(analysis_frames, ignore_index=True)

print(f"Final analysis dataframe shape: {analysis_df.shape}")
print("Columns:")
print(analysis_df.columns.tolist())
print("\nPreview:")
print(analysis_df.head(10).to_string(index=False))